In [1]:
#!pip install python-dotenv

from openai import OpenAI
from pydantic import BaseModel
from dotenv import load_dotenv
import os 

load_dotenv()

# Initialize client
client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

response = client.responses.create(
model="gpt-4o",
input="What is machine learning?"
)

In [2]:
%pip install openai

Note: you may need to restart the kernel to use updated packages.


In [3]:
%pip install datasets

Note: you may need to restart the kernel to use updated packages.


In [4]:
# Install: pip install datasets
from datasets import load_dataset
import requests
from PIL import Image
import pandas as pd
from pathlib import Path
 
# Load dataset from HuggingFace
print("Loading product dataset...")
try:
    # Try loading the dataset
    dataset = load_dataset("ashraq/fashion-product-images-small", split="train[:100]")  # First 100 samples
    print(f"✓ Loaded {len(dataset)} products")
    
    # Convert to pandas for easier manipulation
    products_df = pd.DataFrame(dataset)
    print(f"Dataset columns: {products_df.columns.tolist()}")
    
except Exception as e:
    print(f"⚠ Could not load HuggingFace dataset: {e}")
    print("Using local images instead...")
    
    # Alternative: Use local images
    # Create a products.json file with product information
    products_data = [
        {
            "id": 1,
            "name": "Wireless Headphones",
            "price": 79.99,
            "category": "Electronics",
            "image_path": "images/product1.jpg"
        },
        # Add more products...
    ]
    
    products_df = pd.DataFrame(products_data)
 
# Create images directory
images_dir = Path("product_images")
images_dir.mkdir(exist_ok=True)
 
print(f"\n✓ Dataset prepared!")
print(f"  Total products: {len(products_df)}")

Loading product dataset...


✓ Loaded 100 products
Dataset columns: ['id', 'gender', 'masterCategory', 'subCategory', 'articleType', 'baseColour', 'season', 'year', 'usage', 'productDisplayName', 'image']

✓ Dataset prepared!
  Total products: 100


In [6]:
import base64
 
def encode_image_to_base64(image_path):
    """Encode an image file to base64 string."""
    with open(image_path, "rb") as img_file:
        encoded = base64.b64encode(img_file.read()).decode("utf-8")
    return encoded


In [7]:
def create_product_listing_prompt(product_name, price, category, additional_info=None):
    """
    Create a prompt for generating product listings.
    
    Parameters:
    - product_name: Name of the product
    - price: Price of the product
    - category: Product category
    - additional_info: Optional additional information
    
    Returns:
    - Formatted prompt string
    """
    prompt = f"""You are an expert e-commerce copywriter. Analyze the product image and create a compelling product listing.
 
Product Information:
- Name: {product_name}
- Price: ${price:.2f}
- Category: {category}
{f'- Additional Info: {additional_info}' if additional_info else ''}
 
Please create a professional product listing that includes:
 
1. **Product Title** (catchy, SEO-friendly, 60 characters max)
2. **Product Description** (detailed, 150-200 words)
   - Highlight key features and benefits
   - Use persuasive language
   - Include relevant details visible in the image
3. **Key Features** (bullet points, 5-7 items)
4. **SEO Keywords** (comma-separated, 10-15 relevant keywords)
 
Format your response as JSON with the following structure:
{{
    "title": "Product title here",
    "description": "Full description here",
    "features": ["Feature 1", "Feature 2", ...],
    "keywords": "keyword1, keyword2, ..."
}}
 
Be specific about what you see in the image. Mention colors, materials, design elements, and any distinctive features."""
    
    return prompt
 
# Test prompt creation
test_prompt = create_product_listing_prompt(
    product_name="Wireless Bluetooth Headphones",
    price=79.99,
    category="Electronics",
    additional_info="Noise cancelling, 30-hour battery"
)
 
print("\n" + "="*50)
print("PROMPT TEMPLATE")
print("="*50)
print(test_prompt[:500] + "...")  # Show first 500 characters


PROMPT TEMPLATE
You are an expert e-commerce copywriter. Analyze the product image and create a compelling product listing.

Product Information:
- Name: Wireless Bluetooth Headphones
- Price: $79.99
- Category: Electronics
- Additional Info: Noise cancelling, 30-hour battery

Please create a professional product listing that includes:

1. **Product Title** (catchy, SEO-friendly, 60 characters max)
2. **Product Description** (detailed, 150-200 words)
   - Highlight key features and benefits
   - Use persuasive ...


In [8]:
import base64
import json
from io import BytesIO

# Get the first product and its image from the dataset
product = products_df.iloc[0]
image = product["image"]

# Convert the image to a base64 data URL
image_buffer = BytesIO()
image.convert("RGB").save(image_buffer, format="PNG")
image_base64 = base64.b64encode(image_buffer.getvalue()).decode("utf-8")
image_data_url = f"data:image/png;base64,{image_base64}"

# Include available product details as hints
metadata = {
    "name": str(product.get("productDisplayName", "")),
    "category": str(product.get("masterCategory", "")),
    "subcategory": str(product.get("subCategory", "")),
    "article_type": str(product.get("articleType", "")),
    "color": str(product.get("baseColour", "")),
}

# Require the API response to follow this JSON structure
listing_schema = {
    "type": "object",
    "properties": {
        "title": {"type": "string"},
        "description": {"type": "string"},
        "category": {"type": "string"},
        "color": {"type": "string"},
        "features": {
            "type": "array",
            "items": {"type": "string"},
        },
        "tags": {
            "type": "array",
            "items": {"type": "string"},
        },
    },
    "required": [
        "title",
        "description",
        "category",
        "color",
        "features",
        "tags",
    ],
    "additionalProperties": False,
}

prompt = (
    "Create a concise ecommerce product listing based on the image. "
    "Use the metadata as hints, but do not invent details that cannot be seen. "
    "Return a title, description, category, color, visible features, and search tags.\n\n"
    f"Product metadata: {json.dumps(metadata)}"
)

# Send the prompt and image to the ChatGPT API
response = client.responses.create(
    model="gpt-4o",
    input=[
        {
            "role": "user",
            "content": [
                {"type": "input_text", "text": prompt},
                {"type": "input_image", "image_url": image_data_url},
            ],
        }
    ],
    text={
        "format": {
            "type": "json_schema",
            "name": "product_listing",
            "strict": True,
            "schema": listing_schema,
        }
    },
)

# Check the API response and parse its JSON
print("API response status:", response.status)

if response.status != "completed":
    raise RuntimeError(f"API request did not complete: {response.status}")

product_listing = json.loads(response.output_text)

print("API call succeeded and JSON parsed correctly:")
print(json.dumps(product_listing, indent=2))

API response status: completed
API call succeeded and JSON parsed correctly:
{
  "title": "Turtle Check Men Navy Blue Shirt",
  "description": "Elevate your wardrobe with this Turtle Check Men Navy Blue Shirt. Perfect for casual outings, it features a classic check pattern and a comfortable fit.",
  "category": "Apparel",
  "color": "Navy Blue",
  "features": [
    "Check pattern",
    "Short sleeves",
    "Button-down front"
  ],
  "tags": [
    "men's fashion",
    "shirt",
    "navy blue",
    "check pattern",
    "casual wear"
  ]
}


In [9]:
import base64
import json
from io import BytesIO
from pathlib import Path

MAX_PRODUCTS = 5  # Use None to process every product
OUTPUT_FILE = Path("product_listings.json")

listing_schema = {
    "type": "object",
    "properties": {
        "title": {"type": "string"},
        "description": {"type": "string"},
        "category": {"type": "string"},
        "color": {"type": "string"},
        "features": {"type": "array", "items": {"type": "string"}},
        "tags": {"type": "array", "items": {"type": "string"}},
    },
    "required": [
        "title", "description", "category", "color", "features", "tags"
    ],
    "additionalProperties": False,
}

def make_listing(product):
    """Generate one product listing from its image and available metadata."""
    image = product.get("image")

    if image is None or not hasattr(image, "save"):
        raise ValueError("Product image is missing or is not a PIL image.")

    image_buffer = BytesIO()
    image.convert("RGB").save(image_buffer, format="PNG")
    image_base64 = base64.b64encode(image_buffer.getvalue()).decode("utf-8")
    image_data_url = f"data:image/png;base64,{image_base64}"

    metadata = {
        "name": str(product.get("productDisplayName", "")),
        "category": str(product.get("masterCategory", "")),
        "subcategory": str(product.get("subCategory", "")),
        "article_type": str(product.get("articleType", "")),
        "color": str(product.get("baseColour", "")),
    }

    prompt = (
        "Create a concise ecommerce product listing based on the image. "
        "Use the metadata as hints, but do not invent details that cannot be seen. "
        "Return a title, description, category, color, visible features, and search tags.\n\n"
        f"Product metadata: {json.dumps(metadata)}"
    )

    response = client.responses.create(
        model="gpt-4o",
        input=[
            {
                "role": "user",
                "content": [
                    {"type": "input_text", "text": prompt},
                    {"type": "input_image", "image_url": image_data_url},
                ],
            }
        ],
        text={
            "format": {
                "type": "json_schema",
                "name": "product_listing",
                "strict": True,
                "schema": listing_schema,
            }
        },
    )

    if response.status != "completed":
        raise RuntimeError(f"API request did not complete: {response.status}")

    return json.loads(response.output_text)


# Choose how many products to process
products_to_process = (
    products_df.head(MAX_PRODUCTS)
    if MAX_PRODUCTS is not None
    else products_df
)

results = []

for product_index, product in products_to_process.iterrows():
    result = {
        "product_index": int(product_index),
        "status": None,
    }

    try:
        result["listing"] = make_listing(product)
        result["status"] = "success"
        print(f"✓ Product {product_index}: listing generated")

    except Exception as error:
        result["status"] = "error"
        result["error"] = str(error)
        print(f"✗ Product {product_index}: {error}")

    results.append(result)

    # Save after each product so completed work is kept if a later request fails.
    OUTPUT_FILE.write_text(
        json.dumps(results, indent=2, ensure_ascii=False),
        encoding="utf-8",
    )

successful = sum(item["status"] == "success" for item in results)
failed = sum(item["status"] == "error" for item in results)

print(f"\nProcessed {len(results)} products: {successful} succeeded, {failed} failed.")
print(f"Results saved to: {OUTPUT_FILE.resolve()}")

✓ Product 0: listing generated
✓ Product 1: listing generated
✓ Product 2: listing generated
✓ Product 3: listing generated
✓ Product 4: listing generated

Processed 5 products: 5 succeeded, 0 failed.
Results saved to: /Users/fra/Documents/VScode/lab-api-calling-chatgpt/product_listings.json
